# Retrieval v2 — Colab Pro GPU launcher

## Goal
Notebook này **chỉ là launcher/orchestrator**: mount Google Drive, chuẩn bị source + public dataset, gọi `competition.run_retrieval_v2`, rồi kiểm tra manifest/submission. Toàn bộ thuật toán nằm trong repository.

Luồng chạy gồm 9 stage: `validate-input → keyframes → index → neighbors → segments → text-index → dense-index → predict advanced → validate-submission`. Metric và ground truth không được chạy vì hiện chưa có nhãn. `Experiment.md` chỉ được append sau khi cả 9 stage pass.

## Setup
Bật **Runtime → Change runtime type → GPU** trước khi chạy. Sửa duy nhất cell cấu hình dưới đây. `SOURCE_MODE='drive'` dùng checkout chưa push trong Drive; đổi sang `'git'` sau khi branch đã được push.

In [ ]:
from pathlib import Path

# --- Chỉ sửa block này ---
SOURCE_MODE = 'drive'  # 'drive' hoặc 'git'
GIT_REPO_URL = 'https://github.com/24122013/AIChallenge26_Multimodal_Agentic_Video_Retrieval_System.git'
GIT_BRANCH = 'feat/query-expansion'  # branch/commit phải chứa query expansion mới
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/AIChallenge26_Multimodal_Agentic_Video_Retrieval_System')
PUBLIC_DATA_SOURCE = DRIVE_REPO_PATH / 'data' / 'public'  # thư mục hoặc file .zip
DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/AIChallenge26/retrieval_runs')
MODEL_CACHE_ROOT = Path('/content/drive/MyDrive/AIChallenge26/model_cache')
RUN_ID = 'retrieval-v2-query-expansion-test'  # giữ nguyên để resume; đổi tên cho run mới
START_AT = 'validate-input'
STOP_AFTER = 'validate-submission'
VLM_MODE = 'off'  # off | optional | required; off là cấu hình final ổn định
CAPTION_BATCH_SIZE = 1
CAPTION_QUANTIZATION = '4bit'  # khuyến nghị cho T4/L4; A100 có thể đổi thành 'none'
RRF_K = 60
PADDLE_PACKAGE = 'paddlepaddle-gpu==3.3.0'
PADDLE_INDEX_URL = 'https://www.paddlepaddle.org.cn/packages/stable/cu118/'
REQUIRE_QUERY_EXPANSION = True  # fail notebook nếu provider không chạy thật
DRY_RUN = False

REPO_ROOT = Path('/content/retrieval_repo')
PUBLIC_ROOT = Path('/content/public_data')
RUN_ROOT = DRIVE_RUNS_ROOT / RUN_ID
EXPERIMENT_REPORT = DRIVE_RUNS_ROOT / 'Experiment.md'
print({'run_id': RUN_ID, 'run_root': str(RUN_ROOT), 'public_source': str(PUBLIC_DATA_SOURCE)})


In [ ]:
import os, shlex, shutil, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')

def run_command(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print('$', shlex.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, env=env, check=True)

run_command(['nvidia-smi'])
gpu_line = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total',
    '--format=csv,noheader,nounits',
], text=True).strip().splitlines()[0]
GPU_NAME, GPU_MEMORY_MIB = [value.strip() for value in gpu_line.rsplit(',', 1)]
GPU_MEMORY_MIB = int(GPU_MEMORY_MIB)
if GPU_MEMORY_MIB < 14000:
    raise RuntimeError(f'GPU {GPU_NAME} chỉ có {GPU_MEMORY_MIB} MiB; cần tối thiểu khoảng 14 GiB')
print({'python': sys.version, 'gpu': GPU_NAME, 'gpu_memory_mib': GPU_MEMORY_MIB})


## Step 1 — Chuẩn bị source code
Git mode clone đúng branch. Drive mode copy source sang local SSD của Colab để import/test nhanh hơn, nhưng bỏ qua `.venv` và toàn bộ generated artifact. Thư mục `.git` vẫn được giữ để run manifest ghi đúng commit + dirty diff hash.

In [ ]:
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

if SOURCE_MODE == 'git':
    run_command(['git', 'clone', '--branch', GIT_BRANCH, '--single-branch', GIT_REPO_URL, REPO_ROOT])
elif SOURCE_MODE == 'drive':
    if not (DRIVE_REPO_PATH / '.git').exists():
        raise FileNotFoundError(f'Drive repo phải chứa .git: {DRIVE_REPO_PATH}')
    REPO_ROOT.mkdir(parents=True)
    exclusions = [
        '--exclude=.venv', '--exclude=__pycache__', '--exclude=.pytest_cache',
        '--exclude=data/public', '--exclude=data/model_cache',
        '--exclude=competition/work', '--exclude=competition/keyframes',
        '--exclude=competition/metadata', '--exclude=competition/embeddings',
        '--exclude=competition/indexes', '--exclude=competition/results',
        '--exclude=competition/runs', '--exclude=competition/evaluation',
    ]
    run_command(['rsync', '-a', *exclusions, f'{DRIVE_REPO_PATH}/', f'{REPO_ROOT}/'])
else:
    raise ValueError("SOURCE_MODE phải là 'drive' hoặc 'git'")

run_command(['git', 'status', '--short', '--branch'], cwd=REPO_ROOT)
required = [
    REPO_ROOT / 'competition' / 'run_retrieval_v2.py',
    REPO_ROOT / 'competition' / 'pipeline.py',
    REPO_ROOT / 'configs' / 'retrieval.yaml',
    REPO_ROOT / 'backend' / 'app' / 'services' / 'agent' / 'query_expansion.py',
    REPO_ROOT / 'backend' / 'app' / 'services' / 'retrieval' / 'advanced_search.py',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Source thiếu architecture v2: ' + ', '.join(missing))
retrieval_yaml = (REPO_ROOT / 'configs' / 'retrieval.yaml').read_text(encoding='utf-8')
runner_source = (REPO_ROOT / 'competition' / 'run_retrieval_v2.py').read_text(encoding='utf-8')
pipeline_source = (REPO_ROOT / 'competition' / 'pipeline.py').read_text(encoding='utf-8')
assert 'query_expansion:' in retrieval_yaml and 'enabled: true' in retrieval_yaml
assert '--query-expansion-cache-dir' in runner_source
assert 'precompute_tkis_query_plans' in pipeline_source
print('Source query-expansion contract: OK')


## Step 2 — Cài dependency
Giữ PyTorch CUDA có sẵn của Colab nếu nó đã thỏa constraint; cài `requirements.txt`, cài đúng một PaddlePaddle GPU wheel, rồi kiểm tra Torch CUDA, Paddle CUDA, retrieval config và FFmpeg trước khi inference. Nếu Colab đổi CUDA runtime, cập nhật `PADDLE_INDEX_URL` theo wheel chính thức tương ứng.

In [ ]:
run_command(['apt-get', 'update', '-qq'])
run_command(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'rsync'])
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_ROOT / 'requirements.txt'])
run_command([sys.executable, '-m', 'pip', 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'])
run_command([sys.executable, '-m', 'pip', 'install', '-q', PADDLE_PACKAGE, '-i', PADDLE_INDEX_URL])
run_command([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print('torch', torch.__version__, torch.cuda.get_device_name(0))"])
run_command([sys.executable, '-c', "import paddle; assert paddle.device.is_compiled_with_cuda(); assert paddle.device.cuda.device_count() > 0; print('paddle', paddle.__version__, paddle.device.cuda.device_count())"])
contract_check = (
    "import sys; sys.path.insert(0, " + repr(str(REPO_ROOT)) + " ); "
    "from backend.app.services.retrieval.retrieval_config import load_retrieval_runtime_config; "
    "c=load_retrieval_runtime_config(" + repr(str(REPO_ROOT / 'configs' / 'retrieval.yaml')) + " ); "
    "assert c.query_expansion.enabled and c.query_expansion.max_paraphrases == 2; "
    "print(c.query_expansion.model_name, c.query_expansion.model_revision)"
)
run_command([sys.executable, '-c', contract_check])
run_command(['ffmpeg', '-version'])


## Step 3 — Copy public dataset vào local SSD
`PUBLIC_DATA_SOURCE` có thể là thư mục chứa trực tiếp ba CSV cùng các file được CSV tham chiếu, hoặc file ZIP có một thư mục bao ngoài. Dataset được copy/extract sang `/content/public_data`; artifact và model cache vẫn ghi vào Drive để resume được.

In [ ]:
import zipfile

def source_size_bytes(path):
    if path.is_file() and path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as archive:
            return sum(item.file_size for item in archive.infolist())
    if path.is_dir():
        return sum(item.stat().st_size for item in path.rglob('*') if item.is_file())
    return 0
dataset_bytes = source_size_bytes(PUBLIC_DATA_SOURCE)
local_free = shutil.disk_usage('/content').free
required_free = dataset_bytes + 20 * 1024**3
if local_free < required_free:
    raise RuntimeError(f'Local SSD thiếu chỗ: free={local_free/1024**3:.1f} GiB, cần khoảng {required_free/1024**3:.1f} GiB')
print({'dataset_gib': round(dataset_bytes/1024**3, 2), 'local_free_gib': round(local_free/1024**3, 2)})

if PUBLIC_ROOT.exists():
    shutil.rmtree(PUBLIC_ROOT)
if PUBLIC_DATA_SOURCE.is_dir():
    shutil.copytree(PUBLIC_DATA_SOURCE, PUBLIC_ROOT)
elif PUBLIC_DATA_SOURCE.is_file() and PUBLIC_DATA_SOURCE.suffix.lower() == '.zip':
    PUBLIC_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(PUBLIC_DATA_SOURCE) as archive:
        archive.extractall(PUBLIC_ROOT)
    children = [path for path in PUBLIC_ROOT.iterdir() if path.name != '__MACOSX']
    if not (PUBLIC_ROOT / 'corpus.csv').is_file() and len(children) == 1 and children[0].is_dir():
        nested = children[0]
        for item in list(nested.iterdir()):
            shutil.move(str(item), PUBLIC_ROOT / item.name)
        nested.rmdir()
else:
    raise FileNotFoundError(f'Không tìm thấy public dataset: {PUBLIC_DATA_SOURCE}')

required = ['corpus.csv', 'questions.csv', 'sample_submission.csv']
missing = [name for name in required if not (PUBLIC_ROOT / name).exists()]
if missing:
    raise FileNotFoundError('Public dataset thiếu: ' + ', '.join(missing))
run_command([sys.executable, '-m', 'competition.pipeline', 'validate-input', '--public-root', PUBLIC_ROOT], cwd=REPO_ROOT)


## Step 4 — Chạy end-to-end
Runner tự resume Phase 3, fail-closed nếu code/dataset/offline config khác manifest cũ, giải phóng process/model giữa các stage, build cả coarse index lẫn dense safety index, và dùng advanced retrieval có query expansion mặc định. Notebook truyền caption batch 1 + 4-bit để giảm VRAM trên T4/L4. Nếu Colab bị ngắt, giữ nguyên `RUN_ID`, đặt `START_AT` ở stage cần chạy lại rồi chạy lại notebook.

In [ ]:
DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env.update({
    'PYTHONUTF8': '1',
    'TOKENIZERS_PARALLELISM': 'false',
    'HF_HOME': str(MODEL_CACHE_ROOT / 'huggingface'),
    'TORCH_HOME': str(MODEL_CACHE_ROOT / 'torch'),
    'YOLO_CONFIG_DIR': str(MODEL_CACHE_ROOT / 'ultralytics'),
})
command = [
    sys.executable, '-m', 'competition.run_retrieval_v2',
    '--public-root', PUBLIC_ROOT,
    '--run-root', RUN_ROOT,
    '--experiment-report', EXPERIMENT_REPORT,
    '--experiment-note', 'Colab Pro retrieval v2 E2E; ground truth/metrics unavailable',
    '--device', 'cuda', '--batch-size', 'auto', '--num-workers', '0',
    '--candidate-interval-sec', '0.5', '--max-gap-seconds', '2.0',
    '--target-density-per-second', '0.5',
    '--dedup-similarity-threshold', '0.92',
    '--endpoint-protection', 'on',
    '--model-cache-root', MODEL_CACHE_ROOT,
    '--caption-batch-size', str(CAPTION_BATCH_SIZE),
    '--caption-quantization', CAPTION_QUANTIZATION,
    '--coarse-top-n', '50', '--dense-global-top-k', '300',
    '--dense-rescue-clips', '10', '--max-candidate-clips', '60',
    '--dense-frames-per-clip', '12', '--rrf-k', str(RRF_K),
    '--modality-hint-boost', '1.5', '--vlm-mode', VLM_MODE,
    '--start-at', START_AT, '--stop-after', STOP_AFTER,
]
if REQUIRE_QUERY_EXPANSION and '--no-query-expansion' in command:
    raise ValueError('REQUIRE_QUERY_EXPANSION không cho phép --no-query-expansion')
if DRY_RUN:
    command.append('--dry-run')
run_command(command, cwd=REPO_ROOT, env=env)


## Checks
Cell này không đánh giá chất lượng retrieval. Nó xác nhận architecture contract, submission/checksum và đọc `query_traces.jsonl` để chứng minh production provider đã được gọi cho đủ TKIS query. Nếu `REQUIRE_QUERY_EXPANSION=True`, fallback provider hoặc không có paraphrase hợp lệ sẽ làm cell fail thay vì âm thầm coi original-only là đã test expansion.

In [ ]:
import hashlib, json, yaml

if DRY_RUN:
    print('Dry-run hoàn tất; chưa có artifact để validate.')
else:
    manifest_path = RUN_ROOT / 'run_manifest.json'
    submission_path = RUN_ROOT / 'results' / 'submission.csv'
    trace_path = RUN_ROOT / 'results' / 'query_traces.jsonl'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    expected_stages = ['validate-input', 'keyframes', 'index', 'neighbors', 'segments', 'text-index', 'dense-index', 'predict', 'validate-submission']
    assert manifest['status'] == 'architecture_complete', manifest['status']
    assert all(manifest['stages'][stage]['status'] == 'passed' for stage in expected_stages)
    run_command([sys.executable, '-m', 'competition.pipeline', 'validate-dense-index', '--run-root', RUN_ROOT], cwd=REPO_ROOT)
    run_command([sys.executable, '-m', 'competition.pipeline', 'validate-submission', '--public-root', PUBLIC_ROOT, '--submission-path', submission_path], cwd=REPO_ROOT)
    submission_sha = hashlib.sha256(submission_path.read_bytes()).hexdigest()
    assert manifest['submission']['sha256'] == submission_sha
    traces = [json.loads(line) for line in trace_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    tkis_traces = [row for row in traces if row.get('task') == 'TKIS']
    vkis_traces = [row for row in traces if row.get('task') == 'VKIS']
    assert len(tkis_traces) == 50 and len(vkis_traces) == 50, (len(tkis_traces), len(vkis_traces))
    expansion_rows = [row['query_plan']['query_expansion'] for row in tkis_traces]
    provider_failures = [row for row in expansion_rows if not row.get('provider_name') or row.get('status') != 'passed']
    expanded_rows = [row for row in expansion_rows if any(v.get('accepted') and v.get('type') == 'paraphrase' for v in row.get('variants', []))]
    canonical_failures = [row for row in tkis_traces if row.get('rerank_canonical_query') != row.get('query_plan', {}).get('original_query')]
    vkis_provider_calls = [row for row in vkis_traces if row.get('query_plan', {}).get('query_expansion', {}).get('provider_name')]
    expansion_limit_failures = []
    fusion_cap_failures = []
    metadata_evidence_failures = []
    retrieval_settings = yaml.safe_load((REPO_ROOT / 'configs' / 'retrieval.yaml').read_text(encoding='utf-8'))
    expansion_settings = retrieval_settings['query_expansion']
    expected_expansion_budget = float(expansion_settings['max_expansion_contribution']) * float(expansion_settings['original_weight']) / (float(RRF_K) + 1.0)
    for trace in tkis_traces:
        expansion = trace['query_plan']['query_expansion']
        accepted_paraphrases = [v for v in expansion.get('variants', []) if v.get('accepted') and v.get('type') == 'paraphrase']
        if len(accepted_paraphrases) > 2:
            expansion_limit_failures.append(trace['query_id'])
        for modality, candidates in trace.get('intra_modality_fusion', {}).items():
            for candidate in candidates:
                raw = float(candidate['raw_expansion_contribution'])
                budget = float(candidate['max_expansion_budget'])
                capped = float(candidate['expansion_contribution'])
                final = float(candidate['final_intra_score'])
                original = float(candidate['original_contribution'])
                if abs(budget - expected_expansion_budget) > 1e-10 or abs(capped - min(raw, budget)) > 1e-10 or abs(final - (original + capped)) > 1e-10:
                    fusion_cap_failures.append((trace['query_id'], modality, candidate.get('candidate_id')))
        for result in trace.get('results', []):
            if not {'caption', 'ocr', 'objects'} <= set(result.get('breakdown', {})):
                metadata_evidence_failures.append((trace['query_id'], result.get('candidate_id')))
    if REQUIRE_QUERY_EXPANSION:
        assert not provider_failures, f'Production expansion fallback/disabled: {provider_failures[:3]}'
        assert expanded_rows, 'Provider chạy nhưng không có TKIS query nào giữ được paraphrase hợp lệ'
    assert not canonical_failures, 'Reranker không dùng Original Query'
    assert not vkis_provider_calls, 'VKIS không được gọi query-expansion provider'
    assert not expansion_limit_failures, f'Queries with more than 2 accepted paraphrases: {expansion_limit_failures[:3]}'
    assert not fusion_cap_failures, f'Expansion cap formula mismatch: {fusion_cap_failures[:3]}'
    assert not metadata_evidence_failures, f'Rerank trace missing candidate metadata: {metadata_evidence_failures[:3]}'
    print(json.dumps({
        'status': manifest['status'],
        'run_id': manifest['run_id'],
        'candidate_count': manifest['offline'].get('candidate_count'),
        'selected_count': manifest['offline'].get('selected_count'),
        'submission': str(submission_path),
        'submission_sha256': submission_sha,
        'tkis_provider_passed': len(expansion_rows) - len(provider_failures),
        'tkis_with_accepted_paraphrase': len(expanded_rows),
        'expansion_cap_verified': not fusion_cap_failures,
        'rerank_metadata_verified': not metadata_evidence_failures,
        'query_expansion_verified': REQUIRE_QUERY_EXPANSION and not provider_failures and bool(expanded_rows),
        'experiment_report': str(EXPERIMENT_REPORT),
    }, indent=2))


## Next Steps
1. Nộp file `<RUN_ROOT>/results/submission.csv`.
2. Sau khi có điểm thật, bind điểm vào đúng checksum bằng `python -m competition.pipeline record-score --run-root <RUN_ROOT> --score <SCORE> --split public`.
3. Chỉ promote khi score vượt baseline bằng `python -m competition.pipeline promote-run --run-root <RUN_ROOT> --minimum-score 0.818`.
4. Không đổi config rồi resume cùng `RUN_ID`; hãy tạo run mới để tránh trộn lineage.